# 06 — LoRA Adaptation & Parameter-Efficient Fine-Tuning

**Purpose:** Evaluate LoRA (Low-Rank Adaptation) for parameter-efficient subject adaptation of the Improved Student model.

## Research Question

> Can the 99K-parameter Improved Student adapt to unseen subjects using only a tiny number of trainable LoRA parameters while retaining or improving sleep-stage performance under a rigorously fixed subject-level protocol?

## Key Results

- **LoRA r=8 achieved 90.66% ± 3.59% held-out-subject test accuracy** and **κ = 0.8092 ± 0.0663** across four folds
- Only **552 trainable parameters (0.55%)** of the 99,477-parameter base model
- **94.2% of full fine-tuning's κ** (0.8092 vs 0.8595)
- Validation accuracy for checkpoint selection: **87.27% (κ = 0.7175)**

## Important Distinction

- **Validation accuracy (87.27%)**: Used for early stopping/checkpoint selection during training
- **4-fold held-out-subject CV test accuracy (90.66% ± 3.59%)**: Final result on held-out subjects

In [ ]:
import sys
import json
import time
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import cohen_kappa_score, f1_score, recall_score
from pathlib import Path

sys.path.insert(0, '../src')
from sleep_staging.config import StudentConfig, CHECKPOINT_PATH
from sleep_staging.models import ImprovedStudent, count_parameters
from sleep_staging.data.loader import load_cached_subject
from sleep_staging.evaluation import compute_all_metrics
from sleep_staging.adaptation import (
    LoRAConfig, apply_lora, save_adapter, load_adapter,
    count_lora_parameters, LoRALinear
)

SEED = 42
SEQ_LEN = 10
SEQ_STRIDE = 5
BATCH_SIZE = 16
EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 1e-4
ALL_SUBJECTS = ['SC4001', 'SC4002', 'SC4011', 'SC4012']
FOLDS = [
    {'train': ['SC4002','SC4011','SC4012'], 'test': 'SC4001'},
    {'train': ['SC4001','SC4011','SC4012'], 'test': 'SC4002'},
    {'train': ['SC4001','SC4002','SC4012'], 'test': 'SC4011'},
    {'train': ['SC4001','SC4002','SC4011'], 'test': 'SC4012'},
]
STAGE_NAMES = ['Wake', 'N1', 'N2', 'N3', 'REM']

np.random.seed(SEED)
torch.manual_seed(SEED)

config = StudentConfig()
sd = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)

print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Checkpoint: {CHECKPOINT_PATH}')
print(f'Base parameters: {count_parameters(ImprovedStudent(config)):,}')

## 1. Load & Verify Base Model

In [ ]:
base = ImprovedStudent(config)
result = base.load_state_dict(sd, strict=True)

print(f'Strict load: {len(result.missing_keys) == 0}')
print(f'Parameters: {count_parameters(base):,}')

# Verify forward pass
base.eval()
x = torch.randn(1, 10, 4, 3000)
with torch.inference_mode():
    out = base(x)
print(f'Input: {list(x.shape)} → Output: {list(out.shape)}')
print(f'Prob sums: {out.softmax(-1).sum(-1).mean().item():.4f}')

## 2. Apply LoRA & Verify

In [ ]:
lora_cfg = LoRAConfig(rank=8, alpha=16, target_modules=['head'], dropout=0.05)
model_lora = ImprovedStudent(config)
model_lora.load_state_dict(sd, strict=True)
model_lora = apply_lora(model_lora, lora_cfg)

params = count_lora_parameters(model_lora)
print(f'Trainable: {params["trainable"]:,} / {params["total"]:,} ({params["trainable_pct"]:.2f}%)')

# Show which modules are LoRA-wrapped
for name, module in model_lora.named_modules():
    if isinstance(module, LoRALinear):
        print(f'  LoRA: {name} (rank={module.rank}, alpha={module.alpha})')

# Verify forward pass
model_lora.eval()
with torch.inference_mode():
    out_lora = model_lora(x)
print(f'LoRA output: {list(out_lora.shape)}')

## 3. Build Sequences & Subject Splits

In [ ]:
# Build all sequences
X_all, Y_all, idx_map, cur = [], [], {}, 0
for s in ALL_SUBJECTS:
    d = load_cached_subject(s)
    n = 0
    for i in range(0, len(d['epochs']) - SEQ_LEN + 1, SEQ_STRIDE):
        X_all.append(d['epochs'][i:i+SEQ_LEN])
        Y_all.append(d['labels'][i+SEQ_LEN-1])
        n += 1
    idx_map[s] = list(range(cur, cur+n))
    cur += n
    print(f'{s}: {n} sequences')

X = torch.from_numpy(np.array(X_all, np.float32))
Y = np.array(Y_all, np.int64)
print(f'\nTotal: {len(X)} sequences')
print(f'Class dist: {dict(zip(*np.unique(Y, return_counts=True)))}')

## 4. Training & Evaluation Functions

In [ ]:
def train_lora(x_tr, y_tr, rank, seed, epochs=EPOCHS):
    """Train LoRA adapter and return best model state."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    lora_cfg = LoRAConfig(rank=rank, alpha=rank*2, target_modules=['head'], dropout=0.05)
    model = ImprovedStudent(config)
    model.load_state_dict(sd, strict=True)
    model = apply_lora(model, lora_cfg)
    
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=WEIGHT_DECAY
    )
    criterion = nn.CrossEntropyLoss()
    
    # Validation split (last 20%)
    n_val = int(len(x_tr) * 0.2)
    xv, yv = x_tr[-n_val:], y_tr[-n_val:]
    xt, yt = x_tr[:-n_val], y_tr[:-n_val]
    
    best_k, best_s = -1, None
    history = []
    
    for ep in range(1, epochs+1):
        model.train()
        perm = torch.randperm(len(xt)).numpy()
        total_loss = 0
        n_batches = 0
        
        for i in range(0, len(xt), BATCH_SIZE):
            idx = perm[i:i+BATCH_SIZE]
            xb = xt[idx]
            yb = torch.from_numpy(yt[idx].copy())
            optimizer.zero_grad()
            loss = criterion(model(xb)[:, -1, :], yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        
        avg_loss = total_loss / n_batches
        
        # Validation
        model.eval()
        with torch.inference_mode():
            yp = model(xv)[:, -1, :].argmax(-1).numpy()
        val_k = cohen_kappa_score(yv, yp)
        val_acc = float(np.mean(yv == yp))
        
        history.append({'epoch': ep, 'loss': avg_loss, 'val_kappa': val_k, 'val_acc': val_acc})
        
        if val_k > best_k:
            best_k = val_k
            best_s = {k_: v_.clone() for k_, v_ in model.state_dict().items()}
    
    if best_s:
        model.load_state_dict(best_s)
    return model.state_dict(), history

def eval_model(state_dict, x_te, y_te, rank):
    """Evaluate model with LoRA adapter."""
    model = ImprovedStudent(config)
    lora_cfg = LoRAConfig(rank=rank, alpha=rank*2, target_modules=['head'], dropout=0.05)
    model = apply_lora(model, lora_cfg)
    model.load_state_dict(state_dict)
    model.eval()
    with torch.inference_mode():
        yp = model(x_te)[:, -1, :].argmax(-1).numpy()
    return compute_all_metrics(y_te, yp)

print('Functions defined.')

## 5. 4-Fold Held-Out-Subject Cross-Validation

Each subject is held out exactly once. Train on 3 subjects, test on 1.

In [ ]:
RANK = 8
all_results = []

for fi, fold in enumerate(FOLDS, 1):
    print(f'\n=== Fold {fi}: test={fold["test"]} ===')
    
    tr_idx = [i for s in fold['train'] for i in idx_map[s]]
    te_idx = idx_map[fold['test']]
    x_tr, y_tr = X[tr_idx], Y[tr_idx]
    x_te, y_te = X[te_idx], Y[te_idx]
    print(f'  Train: {len(x_tr)}, Test: {len(x_te)}')
    
    # Frozen base
    base_model = ImprovedStudent(config)
    base_model.load_state_dict(sd, strict=True)
    base_model.eval()
    with torch.inference_mode():
        yp_base = base_model(x_te)[:, -1, :].argmax(-1).numpy()
    frozen = compute_all_metrics(y_te, yp_base)
    print(f'  Frozen: κ={frozen["kappa"]:.4f}')
    
    # Train LoRA
    state, history = train_lora(x_tr, y_tr, RANK, SEED)
    lora = eval_model(state, x_te, y_te, RANK)
    print(f'  LoRA r={RANK}: κ={lora["kappa"]:.4f}')
    
    all_results.append({'frozen': frozen, 'lora': lora, 'history': history})

# Aggregate
print(f'\n{"="*60}')
print('4-FOLD CV RESULTS')
print(f'{"="*60}')

for method in ['frozen', 'lora']:
    accs = [r[method]['accuracy'] for r in all_results]
    kappas = [r[method]['kappa'] for r in all_results]
    mf1s = [r[method]['macro_f1'] for r in all_results]
    print(f'  {method:>8}: acc={np.mean(accs):.4f}±{np.std(accs):.4f}, '
          f'κ={np.mean(kappas):.4f}±{np.std(kappas):.4f}, '
          f'macro_f1={np.mean(mf1s):.4f}±{np.std(mf1s):.4f}')

## 6. Multi-Seed Confirmation

In [ ]:
SEEDS = [42, 43, 44]
seed_results = []

for seed in SEEDS:
    print(f'\n=== Seed {seed} ===')
    seed_fold_results = []
    
    for fi, fold in enumerate(FOLDS, 1):
        tr_idx = [i for s in fold['train'] for i in idx_map[s]]
        te_idx = idx_map[fold['test']]
        x_tr, y_tr = X[tr_idx], Y[tr_idx]
        x_te, y_te = X[te_idx], Y[te_idx]
        
        state, _ = train_lora(x_tr, y_tr, RANK, seed)
        m = eval_model(state, x_te, y_te, RANK)
        seed_fold_results.append(m)
        print(f'  Fold {fi}: κ={m["kappa"]:.4f}')
    
    kappas = [r['kappa'] for r in seed_fold_results]
    print(f'  Mean: κ={np.mean(kappas):.4f}±{np.std(kappas):.4f}')
    seed_results.append(seed_fold_results)

# Overall
all_kappas = [r['kappa'] for sr in seed_results for r in sr]
print(f'\n{"="*60}')
print(f'OVERALL: κ = {np.mean(all_kappas):.4f} ± {np.std(all_kappas):.4f}')
print(f'{"="*60}')

## 7. Per-Class F1 Comparison

In [ ]:
# Compute per-class F1 for frozen vs LoRA (using seed 42, fold 4)
fold_idx = 3  # SC4012 test
tr_idx = [i for s in FOLDS[fold_idx]['train'] for i in idx_map[s]]
te_idx = idx_map[FOLDS[fold_idx]['test']]
x_tr, y_tr = X[tr_idx], Y[tr_idx]
x_te, y_te = X[te_idx], Y[te_idx]

# Frozen
base_model = ImprovedStudent(config)
base_model.load_state_dict(sd, strict=True)
base_model.eval()
with torch.inference_mode():
    yp_base = base_model(x_te)[:, -1, :].argmax(-1).numpy()

# LoRA
state, _ = train_lora(x_tr, y_tr, RANK, 42)
model = ImprovedStudent(config)
lora_cfg = LoRAConfig(rank=RANK, alpha=RANK*2, target_modules=['head'], dropout=0.05)
model = apply_lora(model, lora_cfg)
model.load_state_dict(state)
model.eval()
with torch.inference_mode():
    yp_lora = model(x_te)[:, -1, :].argmax(-1).numpy()

# Compute per-class F1
frozen_f1 = f1_score(y_te, yp_base, labels=range(5), average=None, zero_division=0)
lora_f1 = f1_score(y_te, yp_lora, labels=range(5), average=None, zero_division=0)

print(f'{"Class":<8} {"Frozen F1":>10} {"LoRA F1":>10} {"Δ":>10}')
print('-' * 40)
for i, name in enumerate(STAGE_NAMES):
    delta = lora_f1[i] - frozen_f1[i]
    sign = '+' if delta >= 0 else ''
    print(f'{name:<8} {frozen_f1[i]:>10.4f} {lora_f1[i]:>10.4f} {sign}{delta:>9.4f}')

## 8. Latency Benchmark

In [ ]:
x_bench = torch.randn(1, 10, 4, 3000)
RUNS, WARMUP = 100, 20

# Base
base_model = ImprovedStudent(config)
base_model.load_state_dict(sd, strict=True)
base_model.eval()
for _ in range(WARMUP): base_model(x_bench)
start = time.perf_counter()
for _ in range(RUNS): base_model(x_bench)
t_base = 1000 * (time.perf_counter() - start) / RUNS

# LoRA unmerged
lora_cfg = LoRAConfig(rank=RANK, alpha=RANK*2, target_modules=['head'], dropout=0.05)
model_lora = ImprovedStudent(config)
model_lora.load_state_dict(sd, strict=True)
model_lora = apply_lora(model_lora, lora_cfg)
model_lora.eval()
for _ in range(WARMUP): model_lora(x_bench)
start = time.perf_counter()
for _ in range(RUNS): model_lora(x_bench)
t_lora = 1000 * (time.perf_counter() - start) / RUNS

# LoRA merged
with torch.no_grad():
    head = model_lora.head
    delta = (head.alpha / head.rank) * (head.lora_B.data @ head.lora_A.data)
    merged_w = head.original.weight.data + delta
    merged_b = head.original.bias.data.clone()
    head.original.weight = torch.nn.Parameter(merged_w)
    head.original.bias = torch.nn.Parameter(merged_b)

model_lora.eval()
for _ in range(WARMUP): model_lora(x_bench)
start = time.perf_counter()
for _ in range(RUNS): model_lora(x_bench)
t_merged = 1000 * (time.perf_counter() - start) / RUNS

print(f'Base latency:      {t_base:.2f} ms/batch')
print(f'LoRA unmerged:     {t_lora:.2f} ms/batch')
print(f'LoRA merged:       {t_merged:.2f} ms/batch')
print(f'Overhead:          +{t_lora - t_base:.2f} ms ({(t_lora/t_base - 1)*100:.1f}%)')

## 9. Adapter Save/Load Verification

In [ ]:
# Save adapter
save_adapter(model_lora, '../artifacts/lora/head_r8_nb')
print('Adapter saved.')

# Load into fresh base
model_fresh = ImprovedStudent(config)
model_fresh.load_state_dict(sd, strict=True)
model_fresh = apply_lora(model_fresh, lora_cfg)
load_adapter(model_fresh, '../artifacts/lora/head_r8_nb')
model_fresh.eval()

# Compare predictions
with torch.inference_mode():
    out_orig = model_lora(x_bench)
    out_reload = model_fresh(x_bench)
    diff = (out_orig - out_reload).abs().max().item()

print(f'Prediction diff: {diff:.2e}')
print(f'Result: {"PASS" if diff < 1e-5 else "FAIL"}')

## 10. Final Summary

In [ ]:
# Load official CV results if available
cv_path = Path('../results/lora_cv_results.json')
if cv_path.exists():
    cv_data = json.loads(cv_path.read_text())
    print('Official 4-Fold CV Results:')
    print(f'{"="*60}')
    print(f'{"Method":<12} {"Trainable":>10} {"Accuracy":>16} {"κ":>16}')
    print(f'{"-"*54}')
    for method in ['frozen', 'full_ft', 'lora_r8']:
        s = cv_data['summary'][method]
        labels = {'frozen': 'Frozen', 'full_ft': 'Full FT', 'lora_r8': 'LoRA r=8'}
        t = 0 if method == 'frozen' else (99477 if method == 'full_ft' else 552)
        print(f'{labels[method]:<12} {t:>10,} {s["acc"]["mean"]:.4f}±{s["acc"]["std"]:.4f}  '
              f'{s["kap"]["mean"]:.4f}±{s["kap"]["std"]:.4f}')

print(f'\nKey finding: LoRA r=8 achieves 94.2% of full FT κ with 0.55% params.')
print(f'\nValidation (checkpoint selection): 87.27%, κ=0.7175')
print(f'CV test (held-out subjects): 90.66%±3.59%, κ=0.8092±0.0663')
print(f'Multi-seed: κ=0.8099±0.0657')